# Phase-1 KDE: per-snap optimal α (step function) + E5a base_rate

**Intent:** evaluate phase-1 KDE accuracy at T-5d/T-4d/T-3d/T-2d/T-1d. Compare the **ship stack** (A3 α=0.5, E2) against a **modified stack** with (a) α tuned per-snap and (b) E5a square-root compressed base_rate instead of E2 weighted.

**Phase 1 window (I2):** `(midnight_utc_dbc, snap_dbc_effective]` — reviews between snap and midnight UTC of close day. Excludes close-day post-midnight reviews (those are phase 2 = C=2).

**Ship-stack knobs held fixed:** A3 (σ_gap=8, k=20), B1, C2 (weighted KDE), D2+D3 (bw 0.5-0.7d), F1 (scaling thr=40 clamp (0.5, 2.0)), G2 (midnight snap), H2 (noon-shift), I2, K1. **Varied:** α per snap, and base_rate (E2 vs E5a).

**E5a:** `base_rate = sqrt(Σ w_reviewed / Σ w_all)` (sqrt applied to weighted ratio; preserves C2).

**Decision lens:** per snap, α* = argmin_α MAE; paired bootstrap CI on (E5a@α* − ship).


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd

import _helpers as H
from rotten_tomatoes_forecasting.critic_model import CriticProfiles

print(f'cohort: {len(H.close_date_map)} resolved movies  ·  reviews: {len(H.reviews)}')


## Apply noon-shift (H2)

Shift day-level reviews (timestamp_confidence='d') from midnight UTC to noon UTC. Mutates `H.reviews` in place so all downstream helpers see the shift. Recomputes `H.first_review_ts`, `H.gaps`, and `H.gap_lookup` from the shifted data so the derived artifacts stay consistent.


In [ ]:
_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

# Recompute derived data from the noon-shifted reviews
H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))

print(f'noon-shift: moved {_n_shifted} day-level reviews (+12h), '
      f'left {int((~_day_mask).sum())} h/m reviews unchanged')
print(f'gap_days now  median={H.gaps["gap_days"].median():.2f}  '
      f'IQR={H.gaps["gap_days"].quantile(0.75) - H.gaps["gap_days"].quantile(0.25):.2f}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
ALPHA_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
SIGMA_GAP = 8.0
N_TRAINING = 20
BANDWIDTH_FLOOR = 0.5
BANDWIDTH_CEILING = 0.7
SHRINKAGE_K = 3.0
SCALING_THRESHOLD = 40.0
SCALING_CLAMP = (0.5, 2.0)
MIN_TRAINING_SCORES = 5  # skip if combined_score returned < 5 non-zero candidates (matches noon_shift_test pattern)
CHECKPOINT_EVERY = 500

CACHE_PATH = H.CACHE_DIR / 'phase1_e5a_alpha_step.pkl'
print(f'cache: {CACHE_PATH}')


## E5a profile builder

Drop-in for `H.build_weighted_critic_profiles`. Only change: `base_rate = sqrt(weighted_ratio)`.
Timing data and per-point timing weights preserved for C2 weighted KDE.


In [ ]:
def build_weighted_critic_profiles_e5a(reviews_df, close_date_map_local, training_scores):
    """Like H.build_weighted_critic_profiles but base_rate = sqrt(weighted_ratio)."""
    training_slugs = list(training_scores.keys())
    n_movies = len(training_slugs)
    raw_weights = np.array([training_scores[s] for s in training_slugs], dtype=float)
    total_w = raw_weights.sum()
    if total_w <= 0:
        norm_weights = np.ones_like(raw_weights)
    else:
        norm_weights = raw_weights * (n_movies / total_w)
    slug_weight = dict(zip(training_slugs, norm_weights))

    train = reviews_df[reviews_df['movie_slug'].isin(training_slugs)].copy()
    close_map = pd.Series(close_date_map_local)
    train['bet_close'] = train['movie_slug'].map(close_map)
    train['days_before_close'] = (
        train['bet_close'] - train['estimated_timestamp']
    ).dt.total_seconds() / 86400
    train = train[train['days_before_close'] > 0].copy()
    train['movie_weight'] = train['movie_slug'].map(slug_weight)

    rows = []
    for name, group in train.groupby('reviewer_name'):
        movies_seen = group['movie_slug'].unique()
        weighted_ratio = sum(slug_weight[s] for s in movies_seen) / n_movies
        base_rate = float(np.sqrt(max(weighted_ratio, 0.0)))
        fresh = (group['tomatometer_sentiment'] == 'positive').sum()
        total = len(group)
        rows.append({
            'reviewer_name': name,
            'base_rate': base_rate,
            'fresh_rate': fresh / total if total > 0 else 0.5,
            'timing_data': group['days_before_close'].values.tolist(),
            'timing_weights': group['movie_weight'].values.tolist(),
            'n_reviews': total,
        })

    df = pd.DataFrame(
        rows,
        columns=['reviewer_name', 'base_rate', 'fresh_rate', 'timing_data',
                 'timing_weights', 'n_reviews'],
    )
    return CriticProfiles(df=df, training_slug_count=n_movies)


## Per-(target, snap, α, variant) evaluation

Matches noon_shift_test.ipynb skip-rule convention: snapshot_state + ≥snap_dbc+1 first_review_dbc + ≥3 observed critics + ≥5 training_scores. Wrapped in try/except so rare KDE-fit pathologies can't kill the sweep; failures are counted and sampled.


In [ ]:
_error_counts = {'E2_ship': 0, 'E5a': 0}
_error_samples = []


def eval_target_snap(target_slug, snap_days, alpha, variant):
    try:
        close_ts = H.close_date_map[target_slug]
        midnight_utc = close_ts.floor('D')
        midnight_utc_dbc = (close_ts - midnight_utc).total_seconds() / 86400

        snap_time = midnight_utc - pd.Timedelta(days=snap_days)
        snap_dbc_effective = (close_ts - snap_time).total_seconds() / 86400

        state = H.snapshot_state(target_slug, snap_time)
        ok, _reason = H.passes_skip_rules_for_snap(state, snap_dbc_effective)
        if not ok:
            return None

        target_gap = H.gap_lookup.get(target_slug)
        if target_gap is None:
            return None

        target_window_days = state['first_review_dbc'] - snap_dbc_effective
        if target_window_days <= 0:
            return None

        training_scores = H.combined_score_with_scores(
            target=target_slug,
            target_gap=target_gap,
            target_critics=state['observed_critics'],
            target_window_days=target_window_days,
            k=N_TRAINING,
            alpha=alpha,
            sigma_gap=SIGMA_GAP,
        )
        if len(training_scores) < MIN_TRAINING_SCORES:
            return None

        if variant == 'E2_ship':
            profiles = H.build_weighted_critic_profiles(
                H.reviews, H.close_date_map, training_scores,
            )
        elif variant == 'E5a':
            profiles = build_weighted_critic_profiles_e5a(
                H.reviews, H.close_date_map, training_scores,
            )
        else:
            raise ValueError(variant)

        model = H.build_weighted_kde_lambda_model(
            profiles,
            shrinkage_k=SHRINKAGE_K,
            bandwidth_floor=BANDWIDTH_FLOOR,
            bandwidth_ceiling=BANDWIDTH_CEILING,
        )

        pred = H.predict_window_custom(
            model=model,
            dbc_from=snap_dbc_effective,
            dbc_to=midnight_utc_dbc,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
            scaling_threshold=SCALING_THRESHOLD,
            scaling_clamp=SCALING_CLAMP,
        )

        if not np.isfinite(pred):
            _error_counts[variant] += 1
            if len(_error_samples) < 5:
                _error_samples.append({
                    'target': target_slug, 'snap': snap_days, 'alpha': alpha,
                    'variant': variant, 'reason': f'non-finite pred={pred}',
                })
            return None

        actual = H.actual_in_window(target_slug, snap_dbc_effective, midnight_utc_dbc)

        return {
            'target_slug': target_slug,
            'snap_days': snap_days,
            'alpha': alpha,
            'variant': variant,
            'pred': float(pred),
            'actual': int(actual),
            'err': float(pred) - int(actual),
            'observed_count': state['observed_count'],
            'first_review_dbc': state['first_review_dbc'],
            'target_gap': float(target_gap),
        }
    except Exception as exc:
        _error_counts[variant] = _error_counts.get(variant, 0) + 1
        if len(_error_samples) < 5:
            _error_samples.append({
                'target': target_slug, 'snap': snap_days, 'alpha': alpha,
                'variant': variant, 'reason': f'{type(exc).__name__}: {exc}',
            })
        return None


## Smoke-test on one target


In [ ]:
_smoke_slugs = [s for s in H.close_date_map if s not in {'the_drama', 'the_super_mario_galaxy_movie'}]
_smoke_target = sorted(_smoke_slugs, key=lambda s: H.close_date_map[s], reverse=True)[0]
print(f'smoke target: {_smoke_target}   close={H.close_date_map[_smoke_target]}')
for snap_days in [5, 3, 1]:
    r_ship = eval_target_snap(_smoke_target, snap_days, alpha=0.5, variant='E2_ship')
    r_e5a = eval_target_snap(_smoke_target, snap_days, alpha=0.5, variant='E5a')
    if r_ship is None or r_e5a is None:
        print(f'  T-{snap_days}d: skipped')
        continue
    print(f'  T-{snap_days}d  actual={r_ship["actual"]:3d}  '
          f'ship_pred={r_ship["pred"]:6.2f}  e5a_pred={r_e5a["pred"]:6.2f}  '
          f'ship_err={r_ship["err"]:+6.2f}  e5a_err={r_e5a["err"]:+6.2f}')


## LOO sweep (checkpointed every 500 builds)


In [ ]:
def run_loo(force=False):
    if CACHE_PATH.exists() and not force:
        with open(CACHE_PATH, 'rb') as f:
            cached = pickle.load(f)
        print(f'loaded cached {len(cached)} rows from {CACHE_PATH.name}')
        return cached

    all_targets = sorted(H.close_date_map.keys())
    results = []
    start = time.time()
    total_combos = len(all_targets) * len(SNAP_DAYS_LIST) * (len(ALPHA_GRID) + 1)
    done = 0

    for target_slug in all_targets:
        for snap_days in SNAP_DAYS_LIST:
            r = eval_target_snap(target_slug, snap_days, alpha=0.5, variant='E2_ship')
            if r is not None:
                results.append(r)
            done += 1

            for alpha in ALPHA_GRID:
                r = eval_target_snap(target_slug, snap_days, alpha, variant='E5a')
                if r is not None:
                    results.append(r)
                done += 1

            if done % CHECKPOINT_EVERY == 0:
                elapsed = time.time() - start
                eta = elapsed / done * (total_combos - done)
                print(f'  {done}/{total_combos}  kept {len(results)}  '
                      f'err(E2/E5a)={_error_counts["E2_ship"]}/{_error_counts["E5a"]}  '
                      f'elapsed {elapsed/60:.1f}m  eta {eta/60:.1f}m')
                with open(CACHE_PATH, 'wb') as f:
                    pickle.dump(pd.DataFrame(results), f)

    df = pd.DataFrame(results)
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(df, f)
    print(f'\nsaved {len(df)} rows to {CACHE_PATH}')
    if _error_samples:
        print(f'{sum(_error_counts.values())} errored combos. Samples:')
        for s in _error_samples:
            print(f'  {s}')
    return df

df = run_loo()
print(f'total rows: {len(df)}')
df.head()


## Ship baseline summary (E2, α=0.5)


In [ ]:
def row_metrics(sub):
    err = sub['err'].values
    abs_err = np.abs(err)
    pred = sub['pred'].values
    actual = sub['actual'].values.astype(float)
    safe_actual = np.where(actual > 0, actual, np.nan)
    return {
        'n': len(sub),
        'MAE': float(abs_err.mean()) if len(sub) else np.nan,
        'me': float(err.mean()) if len(sub) else np.nan,
        'med_err': float(np.median(err)) if len(sub) else np.nan,
        'med_abs_err': float(np.median(abs_err)) if len(sub) else np.nan,
        'p90_abs_err': float(np.quantile(abs_err, 0.9)) if len(sub) else np.nan,
        'med_ratio': float(np.nanmedian(pred / safe_actual)) if len(sub) else np.nan,
    }


def print_summary_row(label, m):
    print(f'{label:14s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  '
          f'me={m["me"]:+6.2f}  med_err={m["med_err"]:+6.2f}  '
          f'med|e|={m["med_abs_err"]:6.2f}  p90|e|={m["p90_abs_err"]:6.2f}  '
          f'med_ratio={m["med_ratio"]:5.2f}')

print('=== SHIP BASELINE (A3 α=0.5 + E2 weighted base_rate) ===\n')
ship_metrics = {}
for snap_days in SNAP_DAYS_LIST:
    sub = df[(df['variant'] == 'E2_ship') & (df['snap_days'] == snap_days)]
    m = row_metrics(sub)
    ship_metrics[snap_days] = m
    print_summary_row(f'T-{snap_days}d', m)


## E5a α sweep per snap


In [ ]:
alpha_sweep_rows = []
for snap_days in SNAP_DAYS_LIST:
    for alpha in ALPHA_GRID:
        sub = df[(df['variant'] == 'E5a') & (df['snap_days'] == snap_days) & (df['alpha'] == alpha)]
        if len(sub) == 0:
            continue
        m = row_metrics(sub)
        alpha_sweep_rows.append({'snap_days': snap_days, 'alpha': alpha, **m})
alpha_sweep = pd.DataFrame(alpha_sweep_rows)

for snap_days in SNAP_DAYS_LIST:
    print(f'T-{snap_days}d  (E5a α sweep)')
    snap_df = alpha_sweep[alpha_sweep['snap_days'] == snap_days].copy()
    if snap_df.empty:
        print('  (no data)')
        continue
    argmin_alpha = snap_df.loc[snap_df['MAE'].idxmin(), 'alpha']
    for _, row in snap_df.iterrows():
        flag = '  ← α*' if row['alpha'] == argmin_alpha else ''
        print(f'  α={row["alpha"]:.2f}   MAE={row["MAE"]:6.2f}   '
              f'me={row["me"]:+6.2f}   med_ratio={row["med_ratio"]:.2f}   '
              f'n={int(row["n"]):3d}{flag}')
    print()


## Optimal α step function and side-by-side comparison


In [ ]:
optima_rows = []
for snap_days in SNAP_DAYS_LIST:
    snap_df = alpha_sweep[alpha_sweep['snap_days'] == snap_days]
    if snap_df.empty:
        continue
    best = snap_df.loc[snap_df['MAE'].idxmin()]
    optima_rows.append({
        'snap_days': snap_days,
        'alpha_star': float(best['alpha']),
        'e5a_MAE': float(best['MAE']),
        'e5a_me': float(best['me']),
        'e5a_med_err': float(best['med_err']),
        'e5a_med_ratio': float(best['med_ratio']),
        'e5a_n': int(best['n']),
        'ship_MAE': ship_metrics[snap_days]['MAE'],
        'ship_me': ship_metrics[snap_days]['me'],
        'ship_n': ship_metrics[snap_days]['n'],
    })
optima = pd.DataFrame(optima_rows)
optima['delta_MAE'] = optima['ship_MAE'] - optima['e5a_MAE']
optima['delta_pct'] = 100 * optima['delta_MAE'] / optima['ship_MAE']

print('=== α step function ===\n')
for _, r in optima.iterrows():
    print(f'  T-{int(r["snap_days"])}d:  α* = {r["alpha_star"]:.2f}  '
          f'(MAE {r["e5a_MAE"]:.2f}  vs ship {r["ship_MAE"]:.2f}   '
          f'Δ = {r["delta_pct"]:+.2f}%)')
print()
print('=== side-by-side ===')
optima[['snap_days', 'alpha_star', 'ship_MAE', 'e5a_MAE', 'delta_pct',
        'ship_me', 'e5a_me', 'e5a_med_ratio', 'e5a_n']]


## Paired bootstrap CI (E5a @ α* vs ship)

Per target: delta = |err_ship| − |err_e5a|. Positive = E5a wins. 1000 resamples.


In [ ]:
print('=== paired bootstrap ΔMAE (ship − E5a@α*) ===\n')
print(f'{"snap":<8}{"Δ units":>10}{"CI95_lo":>10}{"CI95_hi":>10}'
      f'{"Δ %":>10}{"CI95_lo %":>12}{"CI95_hi %":>12}{"n":>6}  sig')

ci_rows = []
for _, r in optima.iterrows():
    snap_days = int(r['snap_days'])
    alpha_star = r['alpha_star']
    ship_sub = df[(df['variant'] == 'E2_ship') & (df['snap_days'] == snap_days)]
    e5a_sub = df[(df['variant'] == 'E5a') & (df['snap_days'] == snap_days) & (df['alpha'] == alpha_star)]
    merged = ship_sub.merge(e5a_sub, on='target_slug', suffixes=('_ship', '_e5a'))
    ship_abs = np.abs(merged['err_ship'].values)
    e5a_abs = np.abs(merged['err_e5a'].values)
    deltas = ship_abs - e5a_abs
    if len(deltas) == 0:
        print(f'T-{snap_days}d   (no paired rows)')
        continue
    point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
    ship_mae = ship_abs.mean()
    pct = 100 * point / ship_mae
    lo_pct = 100 * lo / ship_mae
    hi_pct = 100 * hi / ship_mae
    sig = 'SIG' if lo > 0 else ('sig-neg' if hi < 0 else 'ns')
    ci_rows.append({
        'snap_days': snap_days, 'alpha_star': alpha_star,
        'delta': point, 'ci_lo': lo, 'ci_hi': hi,
        'delta_pct': pct, 'ci_lo_pct': lo_pct, 'ci_hi_pct': hi_pct,
        'n_paired': len(merged), 'sig': sig,
    })
    print(f'T-{snap_days}d {point:+10.3f}{lo:+10.3f}{hi:+10.3f}'
          f'{pct:+10.2f}{lo_pct:+12.2f}{hi_pct:+12.2f}{len(merged):6d}  {sig}')
ci_df = pd.DataFrame(ci_rows)


## Plot: MAE vs α per snap


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(SNAP_DAYS_LIST), figsize=(18, 3.6), sharey=False)
for ax, snap_days in zip(axes, SNAP_DAYS_LIST):
    snap_df = alpha_sweep[alpha_sweep['snap_days'] == snap_days].sort_values('alpha')
    if snap_df.empty:
        ax.set_title(f'T-{snap_days}d  (no data)')
        continue
    ax.plot(snap_df['alpha'], snap_df['MAE'], 'o-', color='tab:blue', label='E5a')
    ax.axhline(ship_metrics[snap_days]['MAE'], color='tab:red', linestyle='--',
               label=f'ship α=0.5 E2 (MAE {ship_metrics[snap_days]["MAE"]:.2f})')
    best = snap_df.loc[snap_df['MAE'].idxmin()]
    ax.axvline(best['alpha'], color='tab:green', linestyle=':',
               label=f'α*={best["alpha"]:.2f} (MAE {best["MAE"]:.2f})')
    ax.set_title(f'T-{snap_days}d  (n={int(best["n"])})')
    ax.set_xlabel('α')
    ax.set_ylabel('MAE')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Phase-1 MAE vs α  ·  E5a base_rate  ·  per-snap', y=1.04)
plt.tight_layout()
plt.show()


## Plot: bias (mean error) vs α per snap


In [ ]:
fig, axes = plt.subplots(1, len(SNAP_DAYS_LIST), figsize=(18, 3.6), sharey=False)
for ax, snap_days in zip(axes, SNAP_DAYS_LIST):
    snap_df = alpha_sweep[alpha_sweep['snap_days'] == snap_days].sort_values('alpha')
    if snap_df.empty:
        ax.set_title(f'T-{snap_days}d  (no data)')
        continue
    ax.plot(snap_df['alpha'], snap_df['me'], 'o-', color='tab:blue', label='E5a me')
    ax.axhline(ship_metrics[snap_days]['me'], color='tab:red', linestyle='--',
               label=f'ship me ({ship_metrics[snap_days]["me"]:+.2f})')
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_title(f'T-{snap_days}d')
    ax.set_xlabel('α')
    ax.set_ylabel('mean error (pred − actual)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Phase-1 bias vs α  ·  E5a  ·  per-snap', y=1.04)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
